# E-Commerce Lakehouse: Gold Order Summary

## Aggregate items and payments to order level

Calculate merchandise value, freight value, and item count per order.
Separately calculate recorded payment value per order.

Aggregate before joining to prevent multiple item and payment rows
from inflating totals. Merchandise value excludes freight; recorded
payments are kept as a separate metric.

In [0]:
from pyspark.sql import functions as F

items_df = spark.table("workspace.ecommerce.silver_order_items")
payments_df = spark.table("workspace.ecommerce.silver_payments")

order_item_totals_df = items_df.groupBy("order_id").agg(
    F.count("*").alias("item_count"),
    F.sum("price").alias("merchandise_value"),
    F.sum("freight_value").alias("freight_value")
).withColumn(
    "order_value",
    F.col("merchandise_value") + F.col("freight_value")
)

order_payment_totals_df = payments_df.groupBy("order_id").agg(
    F.count("*").alias("payment_record_count"),
    F.sum("payment_value").alias("recorded_payment_value"),
    F.max(
        F.col("zero_installments_flag").cast("int")
    ).cast("boolean").alias("has_zero_installments"),
    F.max(
        F.col("zero_payment_value_flag").cast("int")
    ).cast("boolean").alias("has_zero_payment"),
    F.max(
        F.col("undefined_payment_type_flag").cast("int")
    ).cast("boolean").alias("has_undefined_payment_type")
)

print("Orders represented in items:", order_item_totals_df.count())
print("Orders represented in payments:", order_payment_totals_df.count())

display(order_item_totals_df.limit(5))
display(order_payment_totals_df.limit(5))

Orders represented in items: 98666
Orders represented in payments: 99440


order_id,item_count,merchandise_value,freight_value,order_value
00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


order_id,payment_record_count,recorded_payment_value,has_zero_installments,has_zero_payment,has_undefined_payment_type
b81ef226f3fe1789b1e8b2acac839d17,1,99.33,false,false,false
a9810da82917af2d9aefd1278f1dcfa0,1,24.39,false,false,false
25e8ea4e93396b6fa0d3dd708e76c1bd,1,65.71,false,false,false
ba78997921bbcdc1373bb41e913ab953,1,107.78,false,false,false
42fdf880ba16b47b59251dd489d4441a,1,128.45,false,false,false


## Combine order-level data

Left join orders to customers, aggregated items, and aggregated payments.
Preserve every order and flag missing item or payment records.

Calculate delivery duration and lateness using calendar dates.
Exclude inconsistent shipping sequences from carrier transit duration.

In [0]:
orders_df = spark.table("workspace.ecommerce.silver_orders")

customers_df = spark.table(
    "workspace.ecommerce.silver_customers"
).select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
)

gold_order_summary_df = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .join(order_item_totals_df, on="order_id", how="left")
    .join(order_payment_totals_df, on="order_id", how="left")
    .withColumn(
        "missing_items_flag",
        F.col("item_count").isNull()
    )
    .withColumn(
        "missing_payments_flag",
        F.col("payment_record_count").isNull()
    )
    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        F.trunc("purchase_date", "month")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "delivery_lateness_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        )
    )
    .withColumn(
        "late_delivery_flag",
        F.when(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull(),
            F.col("delivery_lateness_days") > 0
        )
    )
    .withColumn(
        "carrier_transit_days",
        F.when(
            F.col("shipping_sequence_issue") == False,
            F.datediff(
                "order_delivered_customer_date",
                "order_delivered_carrier_date"
            )
        )
    )
)

gold_order_summary_df.agg(
    F.count("*").alias("order_rows"),
    F.countDistinct("order_id").alias("distinct_order_ids"),
    F.count(
        F.when(F.col("missing_items_flag"), 1)
    ).alias("orders_without_items"),
    F.count(
        F.when(F.col("missing_payments_flag"), 1)
    ).alias("orders_without_payments"),
    F.count(
        F.when(F.col("customer_unique_id").isNull(), 1)
    ).alias("orders_without_customer")
).show(truncate=False)

display(gold_order_summary_df.limit(5))

+----------+------------------+--------------------+-----------------------+-----------------------+
|order_rows|distinct_order_ids|orders_without_items|orders_without_payments|orders_without_customer|
+----------+------------------+--------------------+-----------------------+-----------------------+
|99441     |99441             |775                 |1                      |0                      |
+----------+------------------+--------------------+-----------------------+-----------------------+



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,shipping_sequence_issue,customer_unique_id,customer_city,customer_state,item_count,merchandise_value,freight_value,order_value,payment_record_count,recorded_payment_value,has_zero_installments,has_zero_payment,has_undefined_payment_type,missing_items_flag,missing_payments_flag,purchase_date,purchase_month,delivery_days,delivery_lateness_days,late_delivery_flag,carrier_transit_days
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-10-15T23:32:19.217Z,false,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1,29.99,8.72,38.71,3,38.71,false,false,false,false,false,2017-10-02,2017-10-01,8,-8,false,6
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-10-15T23:32:19.217Z,false,af07308b275d755c9edb36a90c618231,barreiras,BA,1,118.70,22.76,141.46,1,141.46,false,false,false,false,false,2018-07-24,2018-07-01,14,-6,false,12
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-10-15T23:32:19.217Z,false,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,1,159.90,19.22,179.12,1,179.12,false,false,false,false,false,2018-08-08,2018-08-01,9,-18,false,9
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-10-15T23:32:19.217Z,false,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1,45.00,27.20,72.20,1,72.20,false,false,false,false,false,2017-11-18,2017-11-01,14,-13,false,10
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-10-15T23:32:19.217Z,false,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,1,19.90,8.72,28.62,1,28.62,false,false,false,false,false,2018-02-13,2018-02-01,3,-10,false,2


## Reconcile financial totals and save Gold orders

Compare Gold merchandise, freight, and recorded payment totals
with their Silver sources to detect inflation or loss from joins.

Save the order summary as a replaceable reporting table.
Overwrite is intentional: rerunning this step rebuilds the table
rather than appending duplicate orders.

In [0]:
source_items = items_df.agg(
    F.sum("price").alias("merchandise"),
    F.sum("freight_value").alias("freight")
).first()

source_payments = payments_df.agg(
    F.sum("payment_value").alias("payments")
).first()

gold_totals = gold_order_summary_df.agg(
    F.sum("merchandise_value").alias("merchandise"),
    F.sum("freight_value").alias("freight"),
    F.sum("recorded_payment_value").alias("payments")
).first()

for metric, source_value in [
    ("merchandise", source_items["merchandise"]),
    ("freight", source_items["freight"]),
    ("payments", source_payments["payments"])
]:
    gold_value = gold_totals[metric]

    print(f"{metric}: source={source_value}, Gold={gold_value}")

    if source_value != gold_value:
        raise ValueError(f"Financial reconciliation failed: {metric}")

table_name = "workspace.ecommerce.gold_order_summary"

(
    gold_order_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_gold_df = spark.table(table_name)

print("Saved Gold order rows:", saved_gold_df.count())

merchandise: source=13591643.70, Gold=13591643.70
freight: source=2251909.54, Gold=2251909.54
payments: source=16008872.12, Gold=16008872.12
Saved Gold order rows: 99441
